# Shear Peak Counting Statistics In DP1 ECDFS Field

Author: Alex Broughton <br>
Date: 1/27/2026


# Introduction

The goal of this notebook is to decompose the galaxies in the ECDFS field into their reduced shear components and generate statistics of the shear peaks across the field.

Goals:

1. Select metadetect selected galaxies from the ECDFS field.
2. Estimate their shear from their ellipticites.
3. Decompose the ECDFS field into square cells.
4. Construct an optimal smoothing filter and apply it to the cells.
5. Calculate the aperture mass contained in each cell and the distribution of SNR of the peaks.
6. Jacknife re-sample to construct error estimates for the SNR bins.
7. Perform similar steps, randomizing the angle of the galaxies to compare the SNR distribution against a non-lensed null sample of galaxies.

# Setup

In [ ]:
# # ONLY NEED TO RUN ONCE
# ! pip install lenspack

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord
from lsst.daf.butler import Butler
from lsst.rsp import get_tap_service, retrieve_query
import lsst.afw.display as afwDisplay
from astropy.io import fits
from astropy.table import Table
from lenspack.shear import gamma_tx
from scipy.stats import binned_statistic_2d
from tqdm.auto import tqdm
from scipy.ndimage import maximum_filter
import matplotlib.colors as mcolors

butler = Butler("main", collections="u/mgorsuch/ComCam_Cells/ECDFS")
assert butler is not None

service = get_tap_service("tap")
assert service is not None


# Custom color map
colors = ['darkred','red','orange','white','cornflowerblue', 'blue','midnightblue']
cmap = mcolors.LinearSegmentedColormap.from_list('custom_cmap', colors, N=256)

In [ ]:
refs = list(butler.registry.queryDatasets('object_shear', collections="u/mgorsuch/ComCam_Cells/ECDFS"))

In [ ]:
len(refs)

In [ ]:
butler..query_datasets("*shear*", collections="u/mgorsuch/ComCam_Cells/ECDFS")

# 1.0 Gather the data

In [ ]:
# Get data that already exists, or run it again below.
hdul = fits.open('~/DATA/ecdfs_galaxies.fits')
galaxies = Table(hdul[1].data)


In [ ]:
(-28.10-1.5, -28.10+1.5)

In [ ]:
# %%time

# ONLY NEED TO RUN ONCE
(ecdfs_ra, ecdfs_dec) = (53.13, -28.10)
radius_deg = 1.5

query = "SELECT * " \
        "FROM dp1.Object " \
        "WHERE CONTAINS(POINT('ICRS', coord_ra, coord_dec), " \
        "CIRCLE('ICRS', %f, %f, %f)) = 1 " \
        "LIMIT 1" %(ecdfs_ra, ecdfs_dec, radius_deg)

# query = "SELECT objectId, coord_ra, coord_dec, x, y, refExtendedness, " \
#         "g_ixx, g_iyy, g_ixy, r_ixx, r_iyy, r_ixy, i_ixx, i_iyy, i_ixy, " \
#         "g_cModelMag, " \
#         "g_cModelMagErr, " \
#         "r_cModelMag, i_cModelMag, " \
#         "r_cModelMagErr, i_cModelMagErr, " \
#         "r_cModelFlux/r_cModelFluxErr AS r_cModel_SNR, " \
#         "i_cModelFlux/i_cModelFluxErr AS i_cModel_SNR, " \
#         "r_cModel_flag, i_cModel_flag " \
#         "FROM dp1.Object " \
#         "WHERE CONTAINS(POINT('ICRS', coord_ra, coord_dec), " \
#         "CIRCLE('ICRS', %f, %f, %f)) = 1 " %(ecdfs_ra, ecdfs_dec, radius_deg)

job = service.submit_job(query)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()

assert job.phase == 'COMPLETED'
data = job.fetch_result().to_table()
print(len(data))

# for col in data.columns:
#      print(col)

# data.write("~/DATA/ecdfs_objects.fits", format='fits', overwrite=False) 

In [ ]:
data.colnames

In [ ]:
data = Table(fits.open("~/DATA/ecdfs_objects.fits")[1].data)

In [ ]:
data.colnames

# 1.0 Data Reduction

### 1.1 Star/Galaxy Separation

In [ ]:
# Galaxy selection
bands = ["r","i"]

good = data["refExtendedness"] == 1

for band in bands:
    good &= (data[f"{band}_cModel_flag"] == False)

In [ ]:
galaxies = data[good]

In [ ]:
len(galaxies)

In [ ]:
gal_per_deg_sq = len(galaxies) / 1.7**2
gal_per_arcmin_sq = gal_per_deg_sq * (1 / (60./1.)**2)
print(round(gal_per_arcmin_sq, 3), "gal / arcmin^2")
print(round(gal_per_deg_sq, 3), "gal / deg^2")

### 1.1 Calculate $e_1$, $e_2$, $g_1$, $g_2$

In [ ]:
# Construct the needed columns
galaxies.add_column(
    (galaxies["i_ixx"] - galaxies["i_iyy"]) / (galaxies["i_ixx"] + galaxies["i_iyy"]),
    name="i_e1",
)
galaxies.add_column(
    (2*galaxies["i_ixy"]) / (galaxies["i_ixx"] + galaxies["i_iyy"]),
    name="i_e2",
)

galaxies.add_column(
    (galaxies["r_ixx"] - galaxies["r_iyy"]) / (galaxies["r_ixx"] + galaxies["r_iyy"]),
    name="r_e1",
)
galaxies.add_column(
    (2*galaxies["r_ixy"]) / (galaxies["r_ixx"] + galaxies["r_iyy"]),
    name="r_e2",
)
galaxies.add_column(
    galaxies["i_e1"] / 2.,
    name="i_g1",
)
galaxies.add_column(
    galaxies["i_e2"] / 2.,
    name="i_g2",
)

galaxies.add_column(
    galaxies["r_e1"] / 2.,
    name="r_g1",
)
galaxies.add_column(
    galaxies["r_e2"] / 2.,
    name="r_g2",
)

### 1.2 Quality Cuts

In [ ]:
# Good measurements
gooder = ~np.isnan(galaxies['i_e1']) & ~np.isnan(galaxies['i_e2']) & ~np.isnan(galaxies['i_g1']) & ~np.isnan(galaxies['i_g2'])
goodest = gooder & ~np.isnan(galaxies['r_e1']) & ~np.isnan(galaxies['r_e2']) & ~np.isnan(galaxies['r_g1']) & ~np.isnan(galaxies['r_g2'])
galaxies = galaxies[goodest]

In [ ]:
len(galaxies)

In [ ]:
galaxies.write("~/DATA/ecdfs_galaxies.fits", format='fits', overwrite=False) 

## 2.0 Constructing an Optimal Filter

In [ ]:
def Q(theta, theta_max, x_c):
    """
    theta_max (arcmin)
    x_c
    """
    x = theta/theta_max
    f = x / x_c
    
    r = np.tanh(f) / f
    r *= 1 / (1 + np.exp(6-150*theta) + \
              np.exp(-47 + 50*x))
    return r

In [ ]:
theta_max = 3.0 
x_c = 0.15 # 

thetas = np.linspace(1.0e-2, theta_max, 10000)
q = Q(thetas, theta_max=theta_max, x_c=x_c)

In [ ]:
plt.plot(thetas/theta_max, q)
plt.ylim(0,1.15)
plt.xlabel(r"$\theta/\theta_{max}$")
plt.ylabel(r"$Q(\theta)$")


## 3.0 Apply smoothing filter

In [ ]:
# Square bins
nBinsRa = int((np.max(galaxies['coord_ra']) - np.min(galaxies['coord_ra'])) * 3600.0 / 30.0)
nBinsDec = int((np.max(galaxies['coord_dec']) - np.min(galaxies['coord_dec'])) * 3600.0 / 30.0)
e1map, ra_edge, dec_edge, binnumbers = binned_statistic_2d(galaxies['coord_ra'], galaxies['coord_dec'], galaxies["i_e1"], statistic='mean', bins=[nBinsRa, nBinsDec])
e2map, _, _, _ = binned_statistic_2d(galaxies['coord_ra'], galaxies['coord_dec'], galaxies["i_e2"], statistic='mean', bins=[nBinsRa, nBinsDec])
ra_centers = 0.5 * (ra_edge[:-1] + ra_edge[1:])
dec_centers = 0.5 * (dec_edge[:-1] + dec_edge[1:])
rav, decv = np.meshgrid(ra_centers, dec_centers, indexing='xy')


# Construct a WCS for plotting
from astropy.wcs import WCS

wcs = WCS(naxis=2)
wcs.wcs.crpix = [0, 0]  # Reference pixel
wcs.wcs.crval = [float(rav[0][0]), float(decv[0][0])]  # RA, Dec at reference pixel
wcs.wcs.cdelt = [float(np.abs(rav[1][1]-rav[0][0])), float(np.abs(decv[1][1]-decv[0][0]))]
wcs.wcs.ctype = ["RA---TAN", "DEC--TAN"]

# # Plot with WCS
# fig = plt.figure(figsize=(8,8))
# ax = fig.add_subplot(111, projection=wcs)

# # When using WCS projection, DON'T use extent - WCS handles coordinates
# ax.imshow(e1map, origin="lower")  # Remove extent parameter

# # Format coordinates as decimal degrees
# ax.coords[0].set_format_unit('deg')  # RA in degrees
# ax.coords[1].set_format_unit('deg')  # Dec in degrees

# # Optional: control number of decimal places
# ax.coords[0].set_major_formatter('d.dd')  # 2 decimal places
# ax.coords[1].set_major_formatter('d.dd')

# ax.set_xlabel(r'$\alpha$ ($\degree$)')
# ax.set_ylabel(r'$\delta$ ($\degree$)')
# ax.set_aspect(1)

In [ ]:
import healpy as hp

In [ ]:
# Determine appropriate NSIDE for ~30 arcsec resolution
# HEALPix pixel area = 3600 * (4π/12) / nside^2 square arcmin
# For ~30 arcsec pixels: sqrt(3600 * 4π/(12 * (30/60)^2)) ≈ 8192
nside = 8192  # gives ~27 arcsec pixels

phi = galaxies['coord_ra']
theta = galaxies['coord_dec']
pix_indices = hp.ang2pix(nside, theta, phi, nest=False, lonlat=True)

In [ ]:
len(pix_indices)

In [ ]:
len(np.unique(pix_indices))

In [ ]:
# Healpix bins
import healpy as hp

# Determine appropriate NSIDE for ~30 arcsec resolution
# HEALPix pixel area = 3600 * (4π/12) / nside^2 square arcmin
# For ~30 arcsec pixels: sqrt(3600 * 4π/(12 * (30/60)^2)) ≈ 8192
nside = 8192  # gives ~27 arcsec pixels

# Convert RA, Dec to HEALPix pixel indices
phi = np.radians(galaxies['coord_ra'])
theta = np.radians(90.0 - galaxies['coord_dec'])  # convert Dec to colatitude
pix_indices = hp.ang2pix(nside, theta, phi, nest=False)

# Get unique pixels and accumulate statistics
npix = hp.nside2npix(nside)
e1map = np.full(npix, hp.UNSEEN)
e2map = np.full(npix, hp.UNSEEN)

# # Calculate mean for each pixel with galaxies
# unique_pix = np.unique(pix_indices)
# for pix in tqdm(unique_pix, total=len(unique_pix)):
#     mask = pix_indices == pix
#     e1map[pix] = np.mean(galaxies["i_e1"][mask])
#     e2map[pix] = np.mean(galaxies["i_e2"][mask])

# Get RA, Dec centers for pixels with data (if needed for plotting)
theta_centers, phi_centers = hp.pix2ang(nside, unique_pix, nest=False, lonlat=True)
ra_centers = phi_centers
dec_centers = theta_centers)

In [ ]:
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt

# 1. Define the resolution (e.g., NSIDE=32)
# The number of pixels is npix = 12 * nside^2
nside = 32
npix = hp.nside2npix(nside)

# 2. Create some sample data (e.g., a map with random values)
# The data must be a 1D numpy array of size npix
healpix_map = np.random.rand(npix) 

# 3. Plot the map using a Mollweide projection
# The hp.mollview function handles the projection and visualization
hp.mollview(healpix_map, title="Sample HEALPix Map (Mollweide Projection)", unit="Arbitrary Units")


# 4. Display the plot
plt.show()


In [ ]:
hp.mollview?

In [ ]:
# Helpful functions

def find_peaks(dmap, cmap, sn_ratio_map, theta_max, sn_min=None):
    # Apply maximum filter with 3x3 footprint
    max_filtered = maximum_filter(dmap, size=3, mode='constant', cval=-np.inf)
    density = cmap / (np.pi * (theta_max**2))  # theta_max is in arcmin
    
    # Peaks are where the original value equals the local maximum
    peaks = (dmap == max_filtered)
    peaks &= (density > 0.5)
    if sn_min is not None:
        peaks &= (sn_ratio_map>=sn_min)
    
    # Get the indices
    peak_indices = np.where(peaks)
    
    return peaks, peak_indices
    
def get_galaxies_within(xv, theta_max):
    ra, dec = galaxies['coord_ra'], galaxies['coord_dec']
    dra = galaxies['coord_ra'] - xv[0]
    ddec = galaxies['coord_dec'] - xv[1]
    dtheta = np.sqrt(dra**2 + ddec**2)
    bounded = (dtheta <= theta_max)
    galaxies_bounded = galaxies.copy()[bounded]

    return galaxies_bounded , dtheta[bounded], len(galaxies[bounded])


def process_cell(x0, theta_max=1.0, x_c=0.15, randomize=False):
    theta_max *= (1.0/60.0) # Convert from arcmin to degrees
    nearby_galaxies, d_theta, n_g = get_galaxies_within(x0, theta_max=theta_max)

    if n_g == 0:
        return np.nan, np.nan

    xv = nearby_galaxies['coord_ra'], nearby_galaxies['coord_dec']
    e1i = nearby_galaxies['i_g1']
    e2i = nearby_galaxies['i_g2']

    gti, gxi = gamma_tx(
        nearby_galaxies['coord_ra'], 
        nearby_galaxies['coord_dec'],
        nearby_galaxies['i_g1'],
        nearby_galaxies['i_g2'],
        center=x0,
    )

    if randomize:
        rng = np.random.default_rng()
        random_angle = rng.uniform(low=0, high=2 * np.pi)
        gti *= -1 * np.cos(random_angle)
    
    Qi = Q(d_theta, theta_max=theta_max, x_c=x_c)

    ap_mass = (1/n_g) * np.sum(Qi * gti)
    sn_ratio = np.sqrt(2) * ((n_g * ap_mass) / (np.sqrt(np.sum((Qi**2) * (e1i**2 + e2i**2)))))

    return ap_mass, sn_ratio


In [ ]:
len(ra_centers)

In [ ]:
len(ra_centers)

In [ ]:
counts_map = np.full_like(e1map, 0)
for i, ra in tqdm(enumerate(ra_centers), total=len(ra_centers), leave=False):
    for j, dec in tqdm(enumerate(dec_centers), total=len(dec_centers),leave=False):
        x0 = (ra, dec)
        _, _, n = get_galaxies_within(x0, theta_max=1.0)
        counts_map[i][j] = n

In [ ]:
# np.save("/home/alexbroughton/DATA/filtered_shear_maps/counts_map_ECDFS_w_DES_SV_2016_3arcmin_kernel.npy", counts_map)
counts_map = np.load("/home/alexbroughton/DATA/filtered_shear_maps/counts_map_ECDFS_w_DES_SV_2016_3arcmin_kernel.npy")

In [ ]:
gal_per_arcmin2_map = counts_map / (np.pi * (3**2))

In [ ]:
_ = plt.hist(gal_per_arcmin2_map.ravel(), bins='fd', histtype="step")
plt.xlabel(r"gal/arcmin$^2$ within $\theta_{max}$")
plt.ylabel("Counts/cell")
plt.yscale('log')

In [ ]:
%%time

# 8min theta_max=1 arcmin
# 12min 8s theta_max=3 arcmin
# 15min 6s theta_max=12 arcmin

# Run
ap_mass_map = np.full_like(e1map, np.nan)
sn_ratio_map = np.full_like(e1map, np.nan)


for i, ra in tqdm(enumerate(ra_centers), total=len(ra_centers), leave=False):
    for j, dec in tqdm(enumerate(dec_centers), total=len(dec_centers),leave=False):
        x0 = (ra, dec)
        ap_mass, sn_ratio = process_cell(x0, theta_max=3.0, x_c=0.15, randomize=False)
        ap_mass_map[i][j] = ap_mass
        sn_ratio_map[i][j] = sn_ratio

In [ ]:
# Save
np.save("/home/alexbroughton/DATA/filtered_shear_maps/ap_mass_map_ECDFS_w_DES_SV_2016_3arcmin_kernel.npy", ap_mass_map)
np.save("/home/alexbroughton/DATA/filtered_shear_maps/sn_ratio_map_ECDFS_w_DES_SV_2016_3arcmin_kernel.npy", sn_ratio_map)


In [ ]:
# Load from previous run

ap_mass_map = np.load("/home/alexbroughton/DATA/filtered_shear_maps/ap_mass_map_ECDFS_w_DES_SV_2016_3arcmin_kernel.npy")
sn_ratio_map = np.load("/home/alexbroughton/DATA/filtered_shear_maps/sn_ratio_map_ECDFS_w_DES_SV_2016_3arcmin_kernel.npy")

## 3.0 Plot

In [ ]:
peaks, peak_indices = find_peaks(ap_mass_map, counts_map, sn_ratio_map, theta_max=3.0, sn_min=3)
peaks_i = peak_indices[0]
peaks_j = peak_indices[1]

fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(111, projection=wcs)
ax.set_title(r"$\mathcal{S}\;/\mathcal{N} > 3$")
# When using WCS projection, DON'T use extent - WCS handles coordinates
im = ax.imshow(sn_ratio_map, origin="lower", cmap="RdBu_r", vmin=-4, vmax=4)  # Remove extent parameter
ax.scatter(peaks_j, peaks_i, s=200, facecolors='none', 
           edgecolors='black', linewidths=1, marker='o')
plt.colorbar(im, ax=ax)
# Format coordinates as decimal degrees
ax.coords[0].set_format_unit('deg')  # RA in degrees
ax.coords[1].set_format_unit('deg')  # Dec in degrees

# Optional: control number of decimal places
ax.coords[0].set_major_formatter('d.dd')  # 2 decimal places
ax.coords[1].set_major_formatter('d.dd')

ax.set_xlabel(r'$\alpha$ ($\degree$)')
ax.set_ylabel(r'$\delta$ ($\degree$)')
ax.set_aspect(1)



In [ ]:
plt.hist(sn_ratio_map.ravel(), bins='fd')
plt.yscale('log')

In [ ]:
all_peaks, all_peak_indices = find_peaks(ap_mass_map, counts_map, sn_ratio_map, theta_max=3.0, sn_min=None)

_ = plt.hist(sn_ratio_map[all_peaks].ravel(), bins='fd', histtype='step')
plt.title(r"$\theta_{\mathrm{max}} = 3.0$ arcmin")
plt.xlabel(r"$\mathcal{S}\;/\mathcal{N}$")
plt.ylabel("Number of peaks in ECDFS footprint")

In [ ]:
fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(111, projection=wcs)
ax.set_title(r"$M_{ap}$ $(\theta_{\mathrm{max}} = 3$arcmin$)$")
# When using WCS projection, DON'T use extent - WCS handles coordinates
im = ax.imshow(ap_mass_map, origin="lower", cmap="RdBu_r", vmin=-.0025, vmax=.0025)  # Remove extent parameter
plt.colorbar(im, ax=ax)
# Format coordinates as decimal degrees
ax.coords[0].set_format_unit('deg')  # RA in degrees
ax.coords[1].set_format_unit('deg')  # Dec in degrees

# Optional: control number of decimal places
ax.coords[0].set_major_formatter('d.dd')  # 2 decimal places
ax.coords[1].set_major_formatter('d.dd')

ax.set_xlabel(r'$\alpha$ ($\degree$)')
ax.set_ylabel(r'$\delta$ ($\degree$)')
ax.set_aspect(1)



## 4.0 Randomized Analysis

In [ ]:
import os
from glob import glob

base = "/home/alexbroughton/DATA/filtered_shear_maps/randomized"
name = "random_ECDFS_w_DES_SV_2016_3arcmin_kernel"
sn_ratio_fs = []
ap_mass_fs = []

for batch in range(1,3+1):
    for run in range(100):
        sn_ratio_file = os.path.join(base, f"sn_ratio_map_{name}_{batch}_{str(run).zfill(3)}.npy")
        ap_mass_file = os.path.join(base, f"ap_mass_map_{name}_{batch}_{str(run).zfill(3)}.npy")

        if (not os.path.isfile(sn_ratio_file)) or (not os.path.isfile(ap_mass_file)):
            continue
        sn_ratio_fs.append(os.path.join(base, sn_ratio_file))
        ap_mass_fs.append(os.path.join(base, ap_mass_file))


print(len(sn_ratio_fs))

In [ ]:
ap_mass_map_randomized = np.full((len(ap_mass_fs), e1map.shape[0], e1map.shape[1]), np.nan)
sn_ratio_map_randomized = np.full((len(ap_mass_fs), e1map.shape[0], e1map.shape[1]), np.nan)
for i, (ap_mass_file, sn_ratio_file) in tqdm(enumerate(zip(ap_mass_fs, sn_ratio_fs))):
    ap_mass_map_random = np.load(ap_mass_file)
    sn_ratio_map_random = np.load(sn_ratio_file)
    ap_mass_map_randomized[i] = ap_mass_map_random
    sn_ratio_map_randomized[i] = sn_ratio_map_random
    

## 5.0 Plot 

In [ ]:
nBins = 15
sn_min, sn_max = (-0.5, 5)
bins = np.linspace(sn_min, sn_max, nBins + 1)
bin_centers = (bins[:-1] + bins[1:]) / 2

In [ ]:
# Get histogram counts for both arrays using the same bins
peaks, peak_indices = find_peaks(ap_mass_map, counts_map, sn_ratio_map, theta_max=3.0, sn_min=None)
peaks_randomized, peak_indices_randomized = find_peaks(ap_mass_map_randomized[0], counts_map, sn_ratio_map_randomized[0], theta_max=3.0, sn_min=None)

In [ ]:
n_randomized_arr = np.zeros((len(sn_ratio_map_randomized), nBins))
for i in range(len(sn_ratio_map_randomized)):
    peaks_randomized, peak_indices_randomized = find_peaks(ap_mass_map_randomized[i], counts_map, sn_ratio_map_randomized[i], theta_max=3.0, sn_min=None)
    n_randomized_tmp, bins_randomized_tmp, _ = plt.hist(sn_ratio_map_randomized[i][peaks_randomized], bins=bins, range=(sn_min, sn_max), histtype='step')
    n_randomized_arr[i] = n_randomized_tmp

n_randomized = np.mean(n_randomized_arr, axis=0)

In [ ]:
n_data, _ = np.histogram(sn_ratio_map[peaks].ravel(), bins=bins, range=(sn_min, sn_max))
plt.plot(bin_centers, n_data)
plt.plot(bin_centers, n_randomized)

In [ ]:
# Get bin centers for plotting

sigma = np.sqrt((np.std(n_randomized, axis=0)/np.sqrt(300))**2 + counts)
plt.errorbar(bin_centers, n_data - n_randomized, yerr=sigma, marker=".", color="#4285F4",linewidth=1,markersize=10)

plt.axhline(0, linestyle="--", dashes=(8, 10), color="k", linewidth=0.5)
for i in bins:
    plt.axvline(i, color="k", linestyle="dotted", dashes=(3, 3), linewidth=0.5,alpha=0.25)
plt.xlim(-0.33, 5)
plt.xlabel(r"$\mathcal{S}\;/\mathcal{N}$")
plt.ylabel("Number of peaks in ECDFS footprint\n- number of random peaks")


In [ ]:
np.sqrt(counts)

In [ ]:
# Get bin centers for plotting
diff = np.zeros((len(sn_ratio_map_randomized), 15))
for i in range(len(sn_ratio_map_randomized)):
    counts_random, bin_edges = np.histogram(sn_ratio_map_randomized[i].ravel(), bins=bins, range=(sn_min, sn_max))
    counts, _ = np.histogram(sn_ratio_map.ravel(), bins=bins, range=(sn_min, sn_max))
    diff[i] = counts - counts_random

sigma = np.sqrt((np.std(diff, axis=0)/np.sqrt(300))**2 + counts)
plt.errorbar(bin_centers, np.mean(diff, axis=0), yerr=sigma, marker=".", color="#4285F4",linewidth=1,markersize=10)



plt.axhline(0, linestyle="--", dashes=(8, 10), color="k", linewidth=0.5)
for i in bins:
    plt.axvline(i, color="k", linestyle="dotted", dashes=(3, 3), linewidth=0.5,alpha=0.25)
plt.xlim(-0.33, 5)
plt.xlabel(r"$\mathcal{S}\;/\mathcal{N}$")
plt.ylabel("Number of peaks in ECDFS footprint\n- number of random peaks")
